# E1 — structured-check trace and validation

**SESAME claim this notebook serves:** **S1** (`docs/CURRENT.md`).  
**Plan:** `docs/plan/2026-09-01-e1-structured-check-validation.md`.  
**Not this notebook:** human identity correctness (E2), evaluator dependence (E3), or review-time efficiency (E4).

Run **top to bottom** after a fresh kernel. This notebook has two part<path>
1. **Provenance accounting:** explain why the old 41% mixed incompatible units.
2. **New prospective offline trace:** follow every issue-level selected entity from collection-derived structured evidence through candidate retrieval, name-score and temporal checks, final binary decision, and downstream card embedding.

The trace exporter is `experiments/eval/e1_trace_structured_checks.py`. It reads the frozen authority cache only: no live APIs, no language-model calls, no card generation, and no rejection-ledger writes.

Figures and CSV copies go to `analysis/e1_output/`. Frozen JavaScript Object Notation Lines files remain gitignored under `experiments/data/freeze/`.


In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

from IPython.display import display
import matplotlib.pyplot as plt
import pandas as pd


In [2]:
# ── CONFIG (edit this cell; then Run All) ──────────────────────────────
# Default is this GISBG checkout. If you open the notebook from experiments/analysis/
# you can instead set ANALYSIS_DIR = Path(".").resolve()
ANALYSIS_DIR = Path(
    r"<path>"
)
EXPERIMENTS_DIR = ANALYSIS_DIR.parent          # experiments/
REPO_ROOT = EXPERIMENTS_DIR.parent             # jcdl26-newspaper-curation/

# If True, load data/freeze/gate_decisions.jsonl (fast). If missing/False, rebuild
# from v4 cards + ledger via eval/e1_export_gate_table.py (no API calls).
USE_EXISTING_FREEZE = True

# Show the invalid mixed-unit arithmetic so you can see why 41% is retired.
SHOW_INVALID_MIXED_RATE = True

# How many example rows to print per outcome class.
EXAMPLE_N = 3

# Plot
FIGSIZE_BARS = (8.5, 3.8)      # inches
FIG_DPI = 120
BAR_COLOR_OCC = "#5E81AC"
BAR_COLOR_URI = "#81A1C1"
BAR_COLOR_REJ = "#BF616A"
BAR_COLOR_CACHE_OK = "#A3BE8C"
BAR_COLOR_CACHE_EMPTY = "#EBCB8B"
TITLE_FONTSIZE = 12
AXIS_FONTSIZE = 10
ANNOT_FONTSIZE = 9

OUTPUT_DIR = ANALYSIS_DIR / "e1_output"


## Helpers

Pure functions. They read CONFIG names; change display in CONFIG, not here.


In [3]:
def _ensure_experiments_on_path() -> Non<path>    p = str(EXPERIMENTS_DIR)
    if p not in sys.pat<path>        sys.path.insert(0, p)


def load_or_rebuild() -> tuple[pd.DataFrame, dict]:
    freeze = EXPERIMENTS_DIR / "data" / "freeze" / "gate_decisions.jsonl"
    summary_path = EXPERIMENTS_DIR / "artifacts" / "e1_gate_summary.json"
    if USE_EXISTING_FREEZE and freeze.exists():
        rows = [
            json.loads(line)
            for line in freeze.read_text(encoding="utf-8").splitlines()
            if line.strip()
        ]
        df = pd.DataFrame(rows)
        summary = json.loads(summary_path.read_text(encoding="utf-8")) if summary_path.exists() else {}
        return df, summary

    _ensure_experiments_on_path()
    import eval.e1_export_gate_table as e1  # noqa: E402
    e1.main()
    rows = [
        json.loads(line)
        for line in freeze.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
    df = pd.DataFrame(rows)
    summary = json.loads(summary_path.read_text(encoding="utf-8"))
    return df, summary


def recount(df: pd.DataFrame) -> dic<path>    acc = df[df["gate_outcome"] == "accepted"]
    rej = df[df["gate_outcome"] != "accepted"]
    uniq_uri = set(acc["candidate_uri"].dropna().astype(str))
    uniq_rej = set(zip(rej["surface"].fillna(""), rej["candidate_uri"].fillna("")))
    score_u = set(zip(
        rej.loc[rej["gate_outcome"] == "score_rejected", "surface"].fillna(""),
        rej.loc[rej["gate_outcome"] == "score_rejected", "candidate_uri"].fillna(""),
    ))
    temp_u = set(zip(
        rej.loc[rej["gate_outcome"] == "temporal_rejected", "surface"].fillna(""),
        rej.loc[rej["gate_outcome"] == "temporal_rejected", "candidate_uri"].fillna(""),
    ))
    unloc = rej[rej["source"] == "ledger_unlocalized"]
    unloc_pairs = set(zip(unloc["surface"].fillna(""), unloc["candidate_uri"].fillna("")))
    return {
        "accepted_occurrences": int(len(acc)),
        "accepted_unique_uris": len(uniq_uri),
        "accepted_occ_loc": int(acc["candidate_uri"].astype(str).str.contains("id.loc.gov").sum()),
        "accepted_occ_geonames": int(acc["candidate_uri"].astype(str).str.contains("geonames.org").sum()),
        "unique_refusal_pairs": len(uniq_rej),
        "unique_score_refusals": len(score_u),
        "unique_temporal_refusals": len(temp_u),
        "refusal_rows_after_registry_expand": int(len(rej)),
        "refusal_pairs_unlocalized": len(unloc_pairs),
        "n_table_rows": int(len(df)),
    }


## 1 · Load freeze table


In [4]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
df, summary = load_or_rebuild()
counts = recount(df)
print("rows", len(df), "columns", list(df.columns))
print("outcomes", df["gate_outcome"].value_counts().to_dict())
display(pd.Series(counts, name="count").to_frame())
if summar<path>    print("freeze script snapshot cards:", (summary.get("snapshot") or {}).get("cards_per_tier"))


rows 3984 columns ['run_id', 'doc_id', 'dc_field', 'slot', 'surface', 'entity_type', 'candidate_uri', 'gate_outcome', 'reason', 'score', 'source']
outcomes {'accepted': 2837, 'score_rejected': 1070, 'temporal_rejected': 77}


,count
accepted_occurrences,2837
accepted_unique_uris,521
accepted_occ_loc,1923
accepted_occ_geonames,914
unique_refusal_pairs,361
unique_score_refusals,324
unique_temporal_refusals,37
refusal_rows_after_registry_expand,1147
refusal_pairs_unlocalized,2
n_table_rows,3984


freeze script snapshot cards: {'C-simple': 203, 'C-standard': 203, 'C-precedent': 203}


## 2 · Same numbers, three different units

These are **not** interchangeable. The retired “41%” mixed unique refusal *pairs* with unique
accepted *URI strings*.


In [5]:
unit_rows = [
    {
        "unit": "accepted URI occurrences (C-standard card embeds)",
        "n": counts["accepted_occurrences"],
        "what": "one row per URI string found in a C-standard field",
    },
    {
        "unit": "unique accepted URIs",
        "n": counts["accepted_unique_uris"],
        "what": "distinct URI strings among accepted occurrences",
    },
    {
        "unit": "unique ledger (surface, URI) refusals",
        "n": counts["unique_refusal_pairs"],
        "what": "deduped rejection-ledger pairs (324 score + 37 lifespan)",
    },
    {
        "unit": "refusal rows after registry expand",
        "n": counts["refusal_rows_after_registry_expand"],
        "what": "one row per (pair x G0 doc_id); reconstructed, not original",
    },
    {
        "unit": "unlocalized refusal pairs",
        "n": counts["refusal_pairs_unlocalized"],
        "what": "ledger surfaces not in G0 registry (no doc_id)",
    },
]
units = pd.DataFrame(unit_rows)
display(units)
units.to_csv(OUTPUT_DIR / "e1_units.csv", index=False)

fig, ax = plt.subplots(figsize=FIGSIZE_BARS, dpi=FIG_DPI)
labels = ["accepted\noccurrences", "unique\naccepted URIs", "unique\nrefusal pairs"]
vals = [
    counts["accepted_occurrences"],
    counts["accepted_unique_uris"],
    counts["unique_refusal_pairs"],
]
colors = [BAR_COLOR_OCC, BAR_COLOR_URI, BAR_COLOR_REJ]
bars = ax.bar(labels, vals, color=colors)
ax.set_title("E1 counts — different units (do not divide across bars)", fontsize=TITLE_FONTSIZE)
ax.set_ylabel("count", fontsize=AXIS_FONTSIZE)
ax.tick_params(labelsize=AXIS_FONTSIZE)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
for b, v in zip(bars, vals):
    ax.text(
        b.get_x() + b.get_width() / 2, v, f"{v:,}",
        ha="center", va="bottom", fontsize=ANNOT_FONTSIZE,
    )
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "e1_units_bars.png")
plt.show()


,unit,n,what
0,accepted URI occurrences (C-standard card embeds),2837,one row per URI string found in a C-standard f...
1,unique accepted URIs,521,distinct URI strings among accepted occurrences
2,"unique ledger (surface, URI) refusals",361,deduped rejection-ledger pairs (324 score + 37...
3,refusal rows after registry expand,1147,"one row per (pair x G0 doc_id); reconstructed,..."
4,unlocalized refusal pairs,2,ledger surfaces not in G0 registry (no doc_id)


## 3 · The retired mixed-unit rate

Shown only to document the invalid arithmetic. **Do not quote in the paper.**


In [6]:
if SHOW_INVALID_MIXED_RAT<path>    mixed = counts["unique_refusal_pairs"] / (
        counts["accepted_unique_uris"] + counts["unique_refusal_pairs"]
    )
    print(f"361/(521+361) style = {mixed:.4f}  <- INVALID (pairs vs URI strings; ledger vs cards)")
    print("Also invalid because: ledger spans 2026-07-03-07-04; two surfaces unlocalized;")
    print("some URI strings appear on both the accept and refuse sides.")
    print("cache no-hit is a fourth unit (HTTP queries), not a mention count.")


361/(521+361) style = 0.4093  <- INVALID (pairs vs URI strings; ledger vs cards)
Also invalid because: ledger spans 2026-07-03-07-04; two surfaces unlocalized;
some URI strings appear on both the accept and refuse sides.
cache no-hit is a fourth unit (HTTP queries), not a mention count.


## 4 · Cache queries (another unit — not mentions)

`authority_cache.json` is keyed by request URL. Empty hits are not “this mention had no candidate.”


In [7]:
cq = (summary or {}).get("cache_queries") or {}
if not c<path>    _ensure_experiments_on_path()
    import eval.e1_export_gate_table as e1
    cq = e1.cache_query_outcomes()

cache_df = pd.DataFrame([
    {"unit": "cache keys (HTTP queries)", "n": cq.get("n_cache_keys")},
    {"unit": "query_ok (non-empty hits)", "n": cq.get("query_ok")},
    {"unit": "query_no_hit", "n": cq.get("query_no_hit")},
    {"unit": "query_api_error", "n": cq.get("query_api_error")},
])
display(cache_df)
print(cq.get("note"))

fig, ax = plt.subplots(figsize=(6.5, 3.4), dpi=FIG_DPI)
labs = ["ok", "no_hit", "api_error"]
vs = [cq.get("query_ok") or 0, cq.get("query_no_hit") or 0, cq.get("query_api_error") or 0]
ax.bar(labs, vs, color=[BAR_COLOR_CACHE_OK, BAR_COLOR_CACHE_EMPTY, BAR_COLOR_REJ])
ax.set_title("Authority cache — query grain, not mention grain", fontsize=TITLE_FONTSIZE)
ax.set_ylabel("queries", fontsize=AXIS_FONTSIZE)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "e1_cache_queries.png")
plt.show()


,unit,n
0,cache keys (HTTP queries),4634
1,query_ok (non-empty hits),929
2,query_no_hit,3705
3,query_api_error,0


Query-level only; cannot join to (doc_id, mention) without re-running the linker.


## 5 · S1 claim, split into conjuncts

**S1 (CURRENT.md):** *One frozen full-corpus run produces explicit candidate decisions,
gate-accepted URI occurrences, and reasoned refusals.*

That is **three conjuncts plus a scope**. E1 can support some of them fully, some only as a
reconstruction, and some not at all. S1 also does **not** include identity correctness (that
is S2) or a mixed-unit refusal percentage.


In [8]:
S1_SENTENCE = (
    "One frozen full-corpus run produces explicit candidate decisions, "
    "gate-accepted URI occurrences, and reasoned refusals."
)

claim_rows = [
    {
        "piece": "Frozen full-corpus run exists",
        "in_S1_sentence": "yes (scope)",
        "E1_support": "full",
        "evidence": f"{counts['n_table_rows']} reconstructed rows; 203 cards/tier in snapshot",
        "write_in_paper": "v4 N=203, generator gpt-5.2, cards on disk with hashes in freeze manifest",
    },
    {
        "piece": "Gate-accepted URI occurrences",
        "in_S1_sentence": "yes",
        "E1_support": "full (card grain)",
        "evidence": (
            f"{counts['accepted_occurrences']} occ; {counts['accepted_unique_uris']} unique URIs; "
            f"{counts['accepted_occ_loc']} LoC + {counts['accepted_occ_geonames']} GeoNames"
        ),
        "write_in_paper": "C-standard embeds 2,837 gate-accepted URIs (~14/issue; baseline 0). Not human-confirmed identity.",
    },
    {
        "piece": "Reasoned refusals",
        "in_S1_sentence": "yes",
        "E1_support": "partial",
        "evidence": (
            f"{counts['unique_refusal_pairs']} unique (surface, URI); "
            f"{counts['unique_score_refusals']} score + {counts['unique_temporal_refusals']} lifespan; "
            f"{counts['refusal_pairs_unlocalized']} unlocalized"
        ),
        "write_in_paper": "Ledger has 361 unique pair refusals with score/lifespan reasons. Not mention-level; 2 pairs have no doc_id.",
    },
    {
        "piece": "Explicit candidate decisions",
        "in_S1_sentence": "yes",
        "E1_support": "partial / reconstructed",
        "evidence": "Outcomes inferred from card embeds + unique ledger pairs + registry expand. Linker did not freeze mention_id or ranked lists.",
        "write_in_paper": "Call it a reconstructed decision table, not a replay of the live linker.",
    },
    {
        "piece": "Mention-level no_hit / api_error",
        "in_S1_sentence": "implied by 'explicit decisions' in doc 13, not in the one-sentence S1",
        "E1_support": "no (query grain only)",
        "evidence": f"cache no_hit={cq.get('query_no_hit')} / {cq.get('n_cache_keys')} queries",
        "write_in_paper": "Do not treat cache empty-hits as mention NIL counts.",
    },
    {
        "piece": "Ranked candidate lists / Recall@k",
        "in_S1_sentence": "no (doc 13 standard; we lack it)",
        "E1_support": "none",
        "evidence": "cannot_reconstruct includes ranked_candidates",
        "write_in_paper": "Omit Recall@k. Future freeze should store the list.",
    },
    {
        "piece": "Accept/defer/reject (three-way gate)",
        "in_S1_sentence": "no",
        "E1_support": "none",
        "evidence": "authority.py is binary accept vs refuse",
        "write_in_paper": "Do not claim implemented deferral.",
    },
    {
        "piece": "Accepted-link identity precision / refusal correctness",
        "in_S1_sentence": "no — that is S2",
        "E1_support": "none (needs E2 labels)",
        "evidence": "selective-risk CA/IA/CR/IR undefined without gold",
        "write_in_paper": "S1 is auditability of artifacts, not identity validity.",
    },
    {
        "piece": "41% refused / 41% of human work",
        "in_S1_sentence": "no (and invalid)",
        "E1_support": "none",
        "evidence": "mixed units; shown in section 3 only as a counterexample",
        "write_in_paper": "Do not quote.",
    },
]
claim = pd.DataFrame(claim_rows)
display(claim)
claim.to_csv(OUTPUT_DIR / "e1_s1_claim_support.csv", index=False)

print("S1 sentence:")
print(S1_SENTENCE)
print()
print("Supported fully: frozen run + accepted URI occurrences.")
print("Supported partially: reasoned refusals (unique-pair ledger) + explicit decisions (reconstruction).")
print("Not supported by E1: identity correctness, 41%, ranked lists, defer, mention-level NIL.")


,piece,in_S1_sentence,E1_support,evidence,write_in_paper
0,Frozen full-corpus run exists,yes (scope),full,3984 reconstructed rows; 203 cards/tier in sna...,"v4 N=203, generator gpt-5.2, cards on disk wit..."
1,Gate-accepted URI occurrences,yes,full (card grain),2837 occ; 521 unique URIs; 1923 LoC + 914 GeoN...,"C-standard embeds 2,837 gate-accepted URIs (~1..."
2,Reasoned refusals,yes,partial,"361 unique (surface, URI); 324 score + 37 life...",Ledger has 361 unique pair refusals with score...
3,Explicit candidate decisions,yes,partial / reconstructed,Outcomes inferred from card embeds + unique le...,"Call it a reconstructed decision table, not a ..."
4,Mention-level no_hit / api_error,"implied by 'explicit decisions' in doc 13, not...",no (query grain only),cache no_hit=3705 / 4634 queries,Do not treat cache empty-hits as mention NIL c...
5,Ranked candidate lists / Recall@k,no (doc 13 standard; we lack it),none,cannot_reconstruct includes ranked_candidates,Omit Recall@k. Future freeze should store the ...
6,Accept/defer/reject (three-way gate),no,none,authority.py is binary accept vs refuse,Do not claim implemented deferral.
7,Accepted-link identity precision / refusal cor...,no — that is S2,none (needs E2 labels),selective-risk CA/IA/CR/IR undefined without gold,"S1 is auditability of artifacts, not identity ..."
8,41% refused / 41% of human work,no (and invalid),none,mixed units; shown in section 3 only as a coun...,Do not quote.


S1 sentenc<path>One frozen full-corpus run produces explicit candidate decisions, gate-accepted URI occurrences, and reasoned refusals.

Supported fully: frozen run + accepted URI occurrences.
Supported partially: reasoned refusals (unique-pair ledger) + explicit decisions (reconstruction).
Not supported by E1: identity correctness, 41%, ranked lists, defer, mention-level NIL.


### How to read the matrix (plain language)

S1 is an **auditability** claim, not a **correctness** claim.

- **Full:** we can point at frozen cards and count how many times a gate-accepted URI was
  written into C-standard.
- **Partial:** we have a rejection ledger with reasons, but the live run did not store
  `(doc_id, mention)` for refusals. Expanding via the G0 registry is a reconstruction. “Explicit
  candidate decisions” in the *standard* HIPE/selective-prediction sense would include ranked
  lists and mention-level no-hit — we do not have those, so that conjunct of S1 is only weakly
  supported.
- **Not E1:** whether the accepted URI is the right person (S2); any single-number refusal rate;
  three-way defer; human minutes saved.

If Thursday’s long-paper cut is S1+S3+S4 without E2, write S1 as *structure + ledger artifacts*,
never as *verified identities*.


## 6 · Example rows (change `EXAMPLE_N` in CONFIG)


In [9]:
def _show(outcome: str) -> Non<path>    sub = df[df["gate_outcome"] == outcome]
    if outcome != "accepted":
        sub = sub.drop_duplicates(["surface", "candidate_uri"])
    cols = [c for c in [
        "doc_id", "dc_field", "surface", "candidate_uri",
        "gate_outcome", "score", "reason", "source",
    ] if c in sub.columns]
    print(f"=== {outcome} (showing {min(EXAMPLE_N, len(sub))} of {len(sub)}) ===")
    display(sub[cols].head(EXAMPLE_N))


for oc in ["accepted", "score_rejected", "temporal_rejected"]:
    _show(oc)


=== accepted (showing 3 of 2837) ===


,doc_id,dc_field,surface,candidate_uri,gate_outcome,score,reason,source
0,FI21052700_00002,subject,Nightlife,https://id.loc.gov/authorities/subjects/sh2008...,accepted,NaN,,v4_card_embed
1,FI21052700_00002,subject,Entertainment,https://id.loc.gov/authorities/subjects/sh9600...,accepted,NaN,,v4_card_embed
2,FI21052700_00002,subject,Sports,https://id.loc.gov/authorities/subjects/sh0000...,accepted,NaN,,v4_card_embed


=== score_rejected (showing 3 of 324) ===


,doc_id,dc_field,surface,candidate_uri,gate_outcome,score,reason,source
2838,FI21052700_00002,,J. J. Murphy,https://id.loc.gov/authorities/names/nb2008008823,score_rejected,0.689,score 0.689 < threshold 0.75,ledger_x_registry
2839,FI21052700_00002,,E. O. Wilson,https://id.loc.gov/authorities/names/no2015168669,score_rejected,0.717,score 0.717 < threshold 0.75,ledger_x_registry
2840,FI21052700_00002,,Hialeah Park,https://id.loc.gov/authorities/names/n83138890,score_rejected,0.237,score 0.237 < threshold 0.85,ledger_x_registry


=== temporal_rejected (showing 3 of 37) ===


,doc_id,dc_field,surface,candidate_uri,gate_outcome,score,reason,source
2837,FI21052700_00002,,Wilcox,https://id.loc.gov/authorities/names/n2003087428,temporal_rejected,0.689,lifespan: died 1881 < activity window start 1927,ledger_x_registry
2873,FI21052701_00001,,Alexander Robbins,https://id.loc.gov/authorities/names/n85351933,temporal_rejected,0.758,lifespan: died 1922 < activity window start 1949,ledger_x_registry
2874,FI21052701_00001,,W. E. Ellis,https://id.loc.gov/authorities/names/nr99031016,temporal_rejected,0.667,lifespan: died 1876 < activity window start 1949,ledger_x_registry


## 7 · Gaps vs the doc-13 standard freeze frame


In [10]:
standard = [
    "run_id",
    "doc_id",
    "mention_id",
    "normalized_surface",
    "entity_type",
    "ranked_candidates",
    "candidate_uri",
    "score",
    "temporal_state",
    "gate_outcome including defer|no_hit|api_error at mention grain",
    "reason",
    "code_version",
]
have = {
    "run_id": "yes (constant v4-c-standard-2026-07)",
    "doc_id": "yes on accepted; reconstructed or empty on refusals",
    "mention_id": "no (slot on accepted fields only)",
    "normalized_surface": "surface string; registry key used for join",
    "entity_type": "often empty on accepted; present on many ledger rows",
    "ranked_candidates": "no",
    "candidate_uri": "yes",
    "score": "on ledger refusals only",
    "temporal_state": "only as lifespan reason text when refused",
    "gate_outcome including defer|no_hit|api_error at mention grain": (
        "accept / score_rejected / temporal_rejected only; no_hit at cache grain"
    ),
    "reason": "on ledger refusals",
    "code_version": "not stored; infer from repo commit",
}
gap = pd.DataFrame({"standard_field": standard, "in_E1_table": [have[k] for k in standard]})
display(gap)
gap.to_csv(OUTPUT_DIR / "e1_vs_doc13_frame.csv", index=False)
print("Wrote", OUTPUT_DIR)


,standard_field,in_E1_table
0,run_id,yes (constant v4-c-standard-2026-07)
1,doc_id,yes on accepted; reconstructed or empty on ref...
2,mention_id,no (slot on accepted fields only)
3,normalized_surface,surface string; registry key used for join
4,entity_type,often empty on accepted; present on many ledge...
5,ranked_candidates,no
6,candidate_uri,yes
7,score,on ledger refusals only
8,temporal_state,only as lifespan reason text when refused
9,gate_outcome including defer|no_hit|api_error ...,accept / score_rejected / temporal_rejected on...


Wrote <path>


# Part II · Prospective offline structured-check trace

This is the scientific E1. The older card/ledger reconstruction above remains useful provenance, but it does not show how collection-derived structure enters individual checks.

The trace below has one row per **issue-level entity selected by the deployed card pipeline** (up to eight people, organizations, places, and topics per issue). It is not one row per Optical Character Recognition mention.

In [11]:
# Run the offline exporter if the trace or manifest is missing.
TRACE_PATH = EXPERIMENTS_DIR / "data" / "freeze" / "e1_structured_check_trace.jsonl"
TRACE_MANIFEST_PATH = EXPERIMENTS_DIR / "artifacts" / "e1_structured_check_manifest.json"

if not TRACE_PATH.exists() or not TRACE_MANIFEST_PATH.exists():
    _ensure_experiments_on_path()
    import eval.e1_trace_structured_checks as trace_export
    trace_export.main()

trace = pd.DataFrame([
    json.loads(line)
    for line in TRACE_PATH.read_text(encoding="utf-8").splitlines()
    if line.strip()
])
trace_manifest = json.loads(TRACE_MANIFEST_PATH.read_text(encoding="utf-8"))
trace_summary = trace_manifest["summary"]

print("Trace run:", trace_manifest["trace_run_id"])
print("Rows:", len(trace), "Source issues:", trace_summary["n_source_issues"])
print("Offline only:", trace_manifest["offline_only"],
      "Language-model calls:", trace_manifest["llm_calls"],
      "Card generation:", trace_manifest["card_generation"])
print("Trace hash:", trace_manifest["trace_sha256"])
trace.head(3)

Trace run: e1-trace-replay-2026-09-03
Rows: 6401 Source issues: 203
Offline only: True Language-model calls: 0 Card generation: False
Trace hash: 16fce8227539bac35ff9963f823b300249ade5065c0baac170f79d68ae45405b


,trace_run_id,doc_id,entity_rank,surface,normalized_surface,entity_type,g0_registry_key,g0_found,g0_doc_ids,g0_year_min,...,score_threshold,score_state,candidate_birth_year,candidate_death_year,temporal_state,gate_outcome,decision_reason,card_embedded,card_uri_occurrences,card_fields
0,e1-trace-replay-2026-09-03,FI21052700_00002,1,R. Hammerhead Greeby,r hammerhead greeby,person,r hammerhead greeby,True,"[FI21052700_00002, FI21052700_00007, FI2105270...",1927,...,0.75,not_applicable,NaN,NaN,not_applicable,no_hit,no_hit,False,0,[]
1,e1-trace-replay-2026-09-03,FI21052700_00002,2,Little Geraldine,little geraldine,person,little geraldine,True,"[FI21052700_00002, FI21052702_00038, FI2105270...",1927,...,0.75,pass,NaN,NaN,unknown,accept,,True,1,[contributor]
2,e1-trace-replay-2026-09-03,FI21052700_00002,3,Wilcox,wilcox,person,wilcox,True,[FI21052700_00002],1927,...,0.75,fail,1814.0,1881.0,fail,temporal_reject,lifespan: died 1881 < activity window start 1927,False,0,[]


## 8 · Trace completeness

The source has 203 C-standard cards. An issue can legitimately produce no trace rows if the deployed top-entity selector returns nothing. Collection-derived evidence is counted separately from the whole-corpus year fallback.

In [12]:
completeness_rows = [
    ("source issues", trace_summary["n_source_issues"]),
    ("issues with selected entities", trace_summary["n_issues_with_selected_entities"]),
    ("issue-level selected entities", trace_summary["n_rows"]),
    ("entities found in collection registry", trace_summary["g0_registry"].get("found", 0)),
    ("entities not found in collection registry", trace_summary["g0_registry"].get("not_found", 0)),
    ("entity-specific observed-year windows", trace_summary["g0_year_source"].get("entity_observed_years", 0)),
    ("whole-corpus fallback windows", trace_summary["g0_year_source"].get("corpus_fallback", 0)),
]
trace_completeness = pd.DataFrame(completeness_rows, columns=["measure", "count"])
display(trace_completeness)
print("Issues with no selected entities:", trace_summary["issues_with_no_selected_entities"])
print("\nBy entity type:")
display(pd.Series(trace_summary["by_entity_type"], name="selected_entities").to_frame())

# Important interpretation: all topics currently fall outside the person/organization/place registry.
display(pd.crosstab(trace["entity_type"], trace["g0_found"], margins=True))

trace_completeness.to_csv(OUTPUT_DIR / "e1_trace_completeness.csv", index=False)

,measure,count
0,source issues,203
1,issues with selected entities,202
2,issue-level selected entities,6401
3,entities found in collection registry,4796
4,entities not found in collection registry,1605
5,entity-specific observed-year windows,4796
6,whole-corpus fallback windows,1605


Issues with no selected entities: ['FI21052700_00005']

By entity typ<path>

,selected_entities
org,1584
person,1604
place,1608
subject,1605


g0_found,False,True,All
entity_type,,,
org,0,1584,1584
person,0,1604,1604
place,0,1608,1608
subject,1605,0,1605
All,1605,4796,6401


## 9 · Candidate and independent-check yield

The name score is a hand-written lexical similarity in the range zero to one, **not a probability of identity correctness**. Thresholds are 0.75 for people, 0.85 for organizations, and 0.50 for places and topics.

The trace preserves name-score and temporal states independently. This reveals candidates that fail both checks, which the old short-circuiting ledger could not show.

In [13]:
outcome_order = ["accept", "score_reject", "temporal_reject", "no_hit", "cache_error"]
outcome_by_type = pd.crosstab(trace["entity_type"], trace["gate_outcome"]).reindex(
    columns=outcome_order, fill_value=0
)
display(outcome_by_type)

check_summary = pd.DataFrame([
    {"measure": "name-score pass", "count": trace_summary["score_state"].get("pass", 0)},
    {"measure": "name-score fail", "count": trace_summary["score_state"].get("fail", 0)},
    {"measure": "temporal fail (issue-entity rows)", "count": trace_summary["temporal_reject_issue_entity_rows"]},
    {"measure": "temporal fail (unique surface, URI pairs)", "count": trace_summary["temporal_reject_unique_surface_uri_pairs"]},
    {"measure": "fail both name-score and temporal checks", "count": trace_summary["score_and_temporal_fail_overlap"]},
    {"measure": "incremental temporal catches (score passed)", "count": trace_summary["incremental_temporal_catches"]},
])
display(check_summary)

ax = outcome_by_type.plot(
    kind="bar", stacked=True, figsize=(9, 4.2),
    color=["#A3BE8C", "#D08770", "#BF616A", "#EBCB8B", "#4C566A"],
)
ax.set_title("Offline replay outcomes by selected entity type")
ax.set_xlabel("entity type")
ax.set_ylabel("issue-level selected entities")
ax.legend(title="outcome", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "e1_trace_outcomes_by_type.png", dpi=FIG_DPI)
plt.show()

outcome_by_type.to_csv(OUTPUT_DIR / "e1_trace_outcomes_by_type.csv")
check_summary.to_csv(OUTPUT_DIR / "e1_independent_check_yield.csv", index=False)

gate_outcome,accept,score_reject,temporal_reject,no_hit,cache_error
entity_type,,,,,
org,151,442,0,991,0
person,522,124,55,903,0
place,941,138,0,529,0
subject,1334,123,0,148,0


,measure,count
0,name-score pass,2962
1,name-score fail,868
2,temporal fail (issue-entity rows),55
3,"temporal fail (unique surface, URI pairs)",37
4,fail both name-score and temporal checks,41
5,incremental temporal catches (score passed),14


## 10 · What collection-derived time evidence changed

A full system comparison with and without the collection structure is intentionally omitted for this workshop. The minimum mechanism test is narrower: among temporal refusals, how many would have passed the deployed name-score threshold?

Those are **incremental temporal catches**. They show that structured activity windows changed a decision; they do not establish improved identity accuracy until human audit confirms the rejection.

In [14]:
temporal = trace[trace["gate_outcome"] == "temporal_reject"].copy()
incremental = temporal[temporal["score_state"] == "pass"].copy()

print("Temporal-reject issue-entity rows:", len(temporal))
print("Unique temporal-reject (surface, URI) pairs:",
      len(temporal.drop_duplicates(["surface", "candidate_uri"])))
print("Incremental temporal catches that passed name-score threshold:", len(incremental))

cols = [
    "doc_id", "surface", "candidate_label", "candidate_score", "score_threshold",
    "g0_year_min", "g0_year_max", "g0_year_source", "decision_reason", "card_embedded",
]
display(incremental[cols].sort_values(["surface", "doc_id"]).head(20))
incremental[cols].to_csv(OUTPUT_DIR / "e1_incremental_temporal_catches.csv", index=False)

Temporal-reject issue-entity rows: 55
Unique temporal-reject (surface, URI) pairs: 37
Incremental temporal catches that passed name-score threshold: 14


,doc_id,surface,candidate_label,candidate_score,score_threshold,g0_year_min,g0_year_max,g0_year_source,decision_reason,card_embedded
1571,FI21052701_00001,Alexander Robbins,"Robbins, Alexander H. (Alexander Henry), 1875-...",0.758,0.75,1949,1949,entity_observed_years,lifespan: died 1922 < activity window start 1949,False
4836,FI21052703_00002,Charles A. Lindbergh,"Lindbergh, Charles A. (Charles August), 1859-1924",0.762,0.75,1927,1928,entity_observed_years,lifespan: died 1924 < activity window start 1927,False
4196,FI21052702_00033,Charles Laughinghouse,"Laughinghouse, Charles O'Hagan, 1871-1930",0.765,0.75,1935,1935,entity_observed_years,lifespan: died 1930 < activity window start 1935,False
4357,FI21052702_00038,Charles Laughinghouse,"Laughinghouse, Charles O'Hagan, 1871-1930",0.765,0.75,1935,1935,entity_observed_years,lifespan: died 1930 < activity window start 1935,False
2275,FI21052701_00023,Charles Lord,"Cavendish, Charles, Lord, 1704-1783",0.781,0.75,1949,1949,entity_observed_years,lifespan: died 1783 < activity window start 1949,False
3427,FI21052702_00009,Charles R. Myers,"Myers, Charles R. (Charles Richard), 1944-",0.770,0.75,1934,1934,entity_observed_years,lifespan: born 1944 > activity window end 1934,False
2627,FI21052701_00035,Charles Renshaw,"Renshaw, Walter Charles, 1840-1922",0.751,0.75,1949,1949,entity_observed_years,lifespan: died 1922 < activity window start 1949,False
3809,FI21052702_00021,Harold Ross,"Eccles, Harold Ross, 1937-",0.797,0.75,1935,1935,entity_observed_years,lifespan: born 1937 > activity window end 1935,False
4643,FI21052702_00047,Harold Ross,"Eccles, Harold Ross, 1937-",0.797,0.75,1935,1935,entity_observed_years,lifespan: born 1937 > activity window end 1935,False
4706,FI21052702_00049,Harold Ross,"Eccles, Harold Ross, 1937-",0.797,0.75,1935,1935,entity_observed_years,lifespan: born 1937 > activity window end 1935,False


## 11 · Gate acceptance versus final metadata embedding

Passing the gate only makes a URI available to the metadata generator. The generator may omit it; the sanitizer removes URIs outside the allowed set. Therefore accepted candidates and final card URI occurrences must be reported separately.

In [15]:
downstream = pd.DataFrame([
    ("replay-accepted issue-entity candidates", trace_summary["accepted_candidates"]),
    ("accepted candidates whose URI appears in that issue's card", trace_summary["accepted_candidates_embedded_in_card"]),
    ("unique accepted (issue, URI) pairs", trace_summary["accepted_unique_doc_uri_pairs"]),
    ("existing C-standard card URI occurrences", trace_summary["existing_card_uri_occurrences"]),
    ("card URI occurrences covered by replay accepts", trace_summary["existing_card_uri_occurrences_covered_by_replay_accepts"]),
    ("card URI occurrences not covered by replay accepts", trace_summary["existing_card_uri_occurrences_not_covered_by_replay_accepts"]),
], columns=["unit", "count"])
display(downstream)
print("All 2,837 existing card URI occurrences are licensed by an accepted replay candidate:",
      trace_summary["existing_card_uri_occurrences_not_covered_by_replay_accepts"] == 0)
downstream.to_csv(OUTPUT_DIR / "e1_gate_vs_card_embedding.csv", index=False)

,unit,count
0,replay-accepted issue-entity candidates,2948
1,accepted candidates whose URI appears in that ...,2758
2,"unique accepted (issue, URI) pairs",2946
3,existing C-standard card URI occurrences,2837
4,card URI occurrences covered by replay accepts,2837
5,card URI occurrences not covered by replay acc...,0


All 2,837 existing card URI occurrences are licensed by an accepted replay candidate: True


## 12 · Revised S1 support after the new trace

The new trace upgrades E1 from post-hoc accounting to mechanism-level design validation. It still does not supply human identity gold.

In [16]:
revised_claim = pd.DataFrame([
    {
        "claim component": "Frozen corpus-wide trace",
        "status": "supported",
        "evidence": f"{trace_summary['n_rows']:,} issue-entity rows from {trace_summary['n_source_issues']} source issues; hashes recorded",
    },
    {
        "claim component": "Collection-derived structure enters independent checks",
        "status": "supported as design validation",
        "evidence": f"{trace_summary['g0_registry'].get('found', 0):,} rows have entity-registry evidence; name-score and temporal states are separate",
    },
    {
        "claim component": "Structured time evidence changes decisions",
        "status": "supported descriptively",
        "evidence": f"{trace_summary['incremental_temporal_catches']} issue-entity candidates pass score but fail temporal check",
    },
    {
        "claim component": "All final card URIs came from accepted candidates",
        "status": "supported for frozen cards/replay",
        "evidence": f"{trace_summary['existing_card_uri_occurrences_covered_by_replay_accepts']:,}/{trace_summary['existing_card_uri_occurrences']:,} occurrences covered",
    },
    {
        "claim component": "The checks improve identity accuracy",
        "status": "not supported by E1",
        "evidence": "requires E2 human labels and a no-structure comparator for a causal improvement claim",
    },
    {
        "claim component": "Graph representation is superior to a table",
        "status": "not supported",
        "evidence": "deployed temporal check reads the entity registry; it does not traverse the materialized NetworkX graph",
    },
    {
        "claim component": "The gate saves curator time",
        "status": "not supported",
        "evidence": "requires the deferred human time experiment",
    },
])
display(revised_claim)
revised_claim.to_csv(OUTPUT_DIR / "e1_revised_s1_support.csv", index=False)

print("SAFE WORKSHOP CLAIM:")
print("Collection-derived structured context operationalizes independently traceable name-score and temporal checks.")
print("Human identity correctness remains E2; comparative accuracy and curator-time savings are not claimed.")

,claim component,status,evidence
0,Frozen corpus-wide trace,supported,"6,401 issue-entity rows from 203 source issues..."
1,Collection-derived structure enters independen...,supported as design validation,"4,796 rows have entity-registry evidence; name..."
2,Structured time evidence changes decisions,supported descriptively,14 issue-entity candidates pass score but fail...
3,All final card URIs came from accepted candidates,supported for frozen cards/replay,"2,837/2,837 occurrences covered"
4,The checks improve identity accuracy,not supported by E1,requires E2 human labels and a no-structure co...
5,Graph representation is superior to a table,not supported,deployed temporal check reads the entity regis...
6,The gate saves curator time,not supported,requires the deferred human time experiment


SAFE WORKSHOP CLAI<path>Collection-derived structured context operationalizes independently traceable name-score and temporal checks.
Human identity correctness remains E2; comparative accuracy and curator-time savings are not claimed.


## 13 · Candidate traces for human review

These examples are selected by deterministic sorting, not as evidence of accuracy. E2 reviewers must remain blind to machine outcome and score when assigning identity labels.

In [17]:
trace_cols = [
    "doc_id", "surface", "entity_type", "g0_found", "g0_year_min", "g0_year_max",
    "candidate_label", "candidate_uri", "candidate_score", "score_threshold", "score_state",
    "temporal_state", "gate_outcome", "decision_reason", "card_embedded", "card_fields",
]

for outcome in ["accept", "score_reject", "temporal_reject", "no_hit"]:
    sample = trace[trace["gate_outcome"] == outcome].sort_values(
        ["surface", "doc_id"], na_position="last"
    ).head(EXAMPLE_N)
    print(f"\n{outcome}: {len(trace[trace['gate_outcome'] == outcome]):,} total; showing {len(sample)}")
    display(sample[trace_cols])

print("\nE1 complete: trace mechanism validated. Next scientific step is E2 blinded identity audit.")


accept: 2,948 total; showing 3


,doc_id,surface,entity_type,g0_found,g0_year_min,g0_year_max,candidate_label,candidate_uri,candidate_score,score_threshold,score_state,temporal_state,gate_outcome,decision_reason,card_embedded,card_fields
5973,FI21052703_00039,Adam Richards,person,True,1928,1928,"Richards, Adam (Architect)",https://id.loc.gov/authorities/names/nb2017005016,0.766,0.75,pass,unknown,accept,,True,[contributor]
1060,FI21052700_00036,Adele Hampton,person,True,1927,1927,"Hampton, Adele Nilon",https://id.loc.gov/authorities/names/no2008164766,0.765,0.75,pass,unknown,accept,,True,[contributor]
6280,FI21052703_00049,Al Clark,person,True,1928,1928,"Clark, Al",https://id.loc.gov/authorities/names/no2019119007,0.944,0.75,pass,skipped_high_score,accept,,True,[contributor]



score_reject: 827 total; showing 3


,doc_id,surface,entity_type,g0_found,g0_year_min,g0_year_max,candidate_label,candidate_uri,candidate_score,score_threshold,score_state,temporal_state,gate_outcome,decision_reason,card_embedded,card_fields
5716,FI21052703_00030,"""101"" Ranch",org,True,1928,1928,101 Ranch Boys (Musical group),https://id.loc.gov/authorities/names/no2010178827,0.585,0.85,fail,not_applicable,score_reject,score 0.585 < threshold 0.85,False,[]
5303,FI21052703_00017,A. B. C.,org,True,1928,1928,C.E.B.A.,https://id.loc.gov/authorities/names/n96109254,0.500,0.85,fail,not_applicable,score_reject,score 0.5 < threshold 0.85,False,[]
228,FI21052700_00010,Adam McAdam,person,True,1927,1927,"McAdam, Gilbert",https://id.loc.gov/authorities/names/nr92035813,0.355,0.75,fail,unknown,score_reject,score 0.355 < threshold 0.75,False,[]



temporal_reject: 55 total; showing 3


,doc_id,surface,entity_type,g0_found,g0_year_min,g0_year_max,candidate_label,candidate_uri,candidate_score,score_threshold,score_state,temporal_state,gate_outcome,decision_reason,card_embedded,card_fields
6037,FI21052703_00041,A. B. Mack,person,True,1928,1928,"Blackbird, Andrew J., approximately 1820-1908",https://id.loc.gov/authorities/names/no95053489,0.00,0.75,fail,fail,temporal_reject,lifespan: died 1908 < activity window start 1928,False,[]
6069,FI21052703_00042,A. B. Mack,person,True,1928,1928,"Blackbird, Andrew J., approximately 1820-1908",https://id.loc.gov/authorities/names/no95053489,0.00,0.75,fail,fail,temporal_reject,lifespan: died 1908 < activity window start 1928,False,[]
3047,FI21052701_00048,Alex McWilliams,person,True,1949,1949,"McWilliams, Alexander, 1775-1850",https://id.loc.gov/authorities/names/nr2002002374,0.35,0.75,fail,fail,temporal_reject,lifespan: died 1850 < activity window start 1949,False,[]



no_hit: 2,571 total; showing 3


,doc_id,surface,entity_type,g0_found,g0_year_min,g0_year_max,candidate_label,candidate_uri,candidate_score,score_threshold,score_state,temporal_state,gate_outcome,decision_reason,card_embedded,card_fields
5555,FI21052703_00025,"""Georgia"" Meegan",person,True,1928,1928,,,NaN,0.75,not_applicable,not_applicable,no_hit,no_hit,False,[]
3971,FI21052702_00026,"""Hi"" Lawrence",person,True,1935,1935,,,NaN,0.75,not_applicable,not_applicable,no_hit,no_hit,False,[]
5766,FI21052703_00032,"""Pop"" Lehman",person,True,1927,1929,,,NaN,0.75,not_applicable,not_applicable,no_hit,no_hit,False,[]



E1 complete: trace mechanism validated. Next scientific step is E2 blinded identity audit.
